In [1]:
import pandas as pd
import random
import time
random.seed(42)

In [2]:
df = pd.read_csv("DataSchedule.csv")
RUL = pd.read_csv("RUL_consultancy_predictions_A3.csv", sep=";")

## Genetic Algorithm

`load_and_prepare_rul_predictions`: loads a CSV file and returns a dictionary of engine IDs mapped to RUL values.

`compute_safety_due_dates`: computes the safety due date ts = t1 + RUL − 1 for each engine.

`get_engines_in_horizon`: returns all engines whose safety due date is within the planning horizon.

`get_maintenance_duration`: returns the maintenance duration for an engine based on its ID and team type.

`get_team_type`: returns whether a team is type A or type B.

`compute_penalty_cost`: computes the penalty cost for an engine based on how late it is maintained or if it is not maintained.

`create_individual`: creates a random schedule assigning some engines to teams and start days.

`initialize_population`: generates an initial population of random individuals.

`is_feasible`: checks whether a schedule violates constraints such as overlapping jobs or exceeding the horizon.

`compute_total_fitness`: calculates the total penalty cost of a schedule, where lower cost means better fitness.

`parent_selection`: selects parents using tournament selection by choosing the best individual from each sampled group.

`repair_individual`: fixes scheduling conflicts by shifting or removing jobs so that no team has overlapping tasks.

`crossover`: performs single‑point crossover between two parents and repairs the resulting children.

`mutate`: applies a random mutation to one engine and repairs the resulting schedule.


**(a) What does an individual look like, representing a complete maintenance schedule?**

An individual is a fixed-length list of 100 slots, one per engine. Each slot is either `None` (engine not scheduled for maintenance) or a `(team_id, start_day)` tuple. The index in the list corresponds to `engine_id - 1`, so `individual[j]` always refers to engine `j+1`.

**(b) What is your selection strategy (to select parents)?**

Tournament selection: sample k=3 random individuals from the population, select the one with the lowest (best) fitness. Repeat until the desired parent pool size is reached.

**(c) What is your crossover method?**

Single-point crossover on the fixed-length list. A random crossover point is chosen in engines [1, 99]. Child1 takes slots [0:point] from parent1 and [point:] from parent2. Child2 takes the reverse. Because each slot is independent (no duplicates engines possible) only team-overlap conflicts need repair.

**(d) What is your mutation method?**

One of three mutation operators is applied randomly to one engine slot per mutation:
1. **Shift start day**: move the start day of a scheduled engine by a small random amount.
2. **Swap team**: reassign the engine to a different randomly chosen team.
3. **Toggle schedule**: add an unscheduled engine to the schedule, or remove a scheduled one.

**(e) How do you deal with infeasible individuals after crossover and/or mutation?**

A `repair_individual` function is called after every crossover and mutation. It resolves team scheduling conflicts by iterating over each team's jobs sorted by start day, and pushing any overlapping job to start immediately after the previous one ends. Jobs that cannot fit within T are removed (set to None).

**(f) Hyperparameters and design choices?**
-initializing the population with 30% change per engine in `create_individual`

In [ ]:
M = 100        # total number of engines
G = 4          # total number of teams
T = 30         # planning horizon in days
t1 = 1         # current day
TEAMS = ["T1", "T2", "T3", "T4"]

def load_and_prepare_rul_predictions(filepath):
    """
    Load RUL predictions from a CSV file and return a dictionary mapping
    engine_id to predicted RUL value (rounded to integer).
    """
    df = pd.read_csv(filepath, sep=";")
    df["RUL"] = df["RUL"].astype(int)
    rul_dict = dict(zip(df["id"], df["RUL"]))
    return rul_dict


def compute_safety_due_dates(rul_predictions, t1=1):
    """
    Compute the safety due date ts = t1 + RUL - 1 for each engine.
    Returns a dict mapping engine_id -> safety due date.
    """
    safety_due_dates = {}
    for engine_id, rul in rul_predictions.items():
        safety_due_dates[engine_id] = t1 + rul - 1
    return safety_due_dates


def get_engines_in_horizon(safety_due_dates, T=30):
    """
    Return the list of engine_ids whose safety due date falls within
    the planning horizon (ts <= T).
    """
    engines_in_horizon = [
        engine_id for engine_id, ts in safety_due_dates.items() if ts <= T
    ]
    return engines_in_horizon

def get_maintenance_duration(engine_id, team_type):
    """
    Return maintenance duration (days) for a given engine and team type (A or B).
    Based on mu_A and mu_B rules from the assignment.
    """
    # mu_A by engine range
    if engine_id <= 20:
        mu_A = 5
    elif engine_id <= 55:
        mu_A = 3
    elif engine_id <= 80:
        mu_A = 4
    else:
        mu_A = 5

    # mu_B by engine range
    if engine_id <= 25:
        mu_B = mu_A - 1
    elif engine_id <= 70:
        mu_B = mu_A + 3
    else:
        mu_B = mu_A + 2

    return mu_A if team_type == "A" else mu_B


def get_team_type(team_id):
    """Return 'A' for teams T1/T3 and 'B' for teams T2/T4."""
    return "A" if team_id in ["T1", "T3"] else "B"


def compute_penalty_cost(engine_id, maintenance_end_day, safety_due_date, T=30):
    """
    Compute the total penalty cost for an engine.
    - If maintenance_end_day is None: engine is unmaintained, penalise every
      day after ts up to T.
    - Otherwise: penalise every day after ts up to maintenance_end_day.
    Daily cost = cj * (t - ts)^2, capped at 250.
    """
    # Cost coefficient cj by engine range
    if engine_id <= 25:
        cj = 4
    elif engine_id <= 45:
        cj = 2
    elif engine_id <= 75:
        cj = 5
    else:
        cj = 6

    total_cost = 0
    last_late_day = T if maintenance_end_day is None else maintenance_end_day

    for t in range(safety_due_date + 1, last_late_day + 1):
        daily_cost = cj * (t - safety_due_date) ** 2
        total_cost += min(daily_cost, 250)

    return total_cost

# Individual: fixed-length list of length M

# individual[j] corresponds to engine (j + 1), j in range(M), due to python indexation
# individual[j] = None                 -> engine not scheduled
# individual[j] = (team_id, start_day) -> engine scheduled

def create_individual(engines_in_horizon, T=30):
    """
    Create a random individual as a fixed-length list of M slots.
    Only engines in the horizon are candidates for scheduling.
    Each candidate is scheduled with 30% probability to keep the
    initial population sparse and easier to repair.
    """
    # Start with all slots as None (no maintenance scheduled)
    individual = [None] * M

    horizon_set = set(engines_in_horizon)

    for engine_id in range(1, M + 1):
        # Only consider engines whose safety due date is within the horizon
        if engine_id not in horizon_set:
            continue

        # Schedule the engines with 30% probability
        if random.random() > 0.30:
            continue

        # Assign to a team
        team_id = random.choice(TEAMS)
        team_type = get_team_type(team_id)
        duration = get_maintenance_duration(engine_id, team_type)

        # Pick a valid start day so maintenance finishes within horizon
        latest_start = T - duration + 1
        if latest_start < 1:
            continue

        start_day = random.randint(1, latest_start)
        individual[engine_id - 1] = (team_id, start_day)

    return individual

def initialize_population(engines_in_horizon, population_size, T=30):
    """
    Generate an initial population of fixed-length individuals.
    Returns a list of population_size individuals.
    """
    return [create_individual(engines_in_horizon, T) for _ in range(population_size)]

def is_feasible(individual, T=30):
    """
    Check that an individual satisfies all constraints:
    - No team has overlapping jobs.
    - All jobs complete within the planning horizon T.
    Duplicate engines are impossible by design (fixed-length list).
    Returns True if feasible, False otherwise.
    """
    # Track scheduled intervals per team: {team_id: [(start, end), ...]}
    team_schedules = {team: [] for team in TEAMS}

    for engine_id in range(1, M + 1):
        slot = individual[engine_id - 1]
        if slot is None:
            continue

        team_id, start_day = slot
        duration = get_maintenance_duration(engine_id, get_team_type(team_id))
        end_day = start_day + duration - 1

        # Job must complete within horizon
        if end_day > T:
            return False

        # Team must not already be busy during this interval
        for existing_start, existing_end in team_schedules[team_id]:
            if start_day <= existing_end and end_day >= existing_start:
                return False

        team_schedules[team_id].append((start_day, end_day))

    return True

def compute_total_fitness(individual, safety_due_dates, engines_in_horizon, T=30):
    """
    Compute the total penalty cost for a given individual.
    For each engine in the horizon:
    - If scheduled: penalise based on maintenance end day vs safety due date.
    - If not scheduled: penalise as if unmaintained for the full horizon.
    Lower fitness = better.
    """
    total_cost = 0

    for engine_id in engines_in_horizon:
        slot = individual[engine_id - 1]
        ts = safety_due_dates[engine_id]

        if slot is not None:
            team_id, start_day = slot
            duration = get_maintenance_duration(engine_id, get_team_type(team_id))
            end_day = start_day + duration - 1
            total_cost += compute_penalty_cost(engine_id, end_day, ts, T)
        else:
            # Engine is not maintained: full horizon penalty
            total_cost += compute_penalty_cost(engine_id, None, ts, T)

    return total_cost

def parent_selection(population, fitnesses, parent_size, tournament_size=3):
    """
    Tournament selection: randomly sample tournament_size individuals,
    keep the one with the lowest fitness. Repeat until parent_size winners
    are collected. Selection is with replacement.
    """
    winners = []

    for _ in range(parent_size):
        # Sample random indices for the tournament group
        tournament_indices = random.sample(range(len(population)), tournament_size)

        # Find the index with the best (lowest) fitness
        best_index = min(tournament_indices, key=lambda i: fitnesses[i])
        winners.append(population[best_index])

    return winners

def repair_individual(individual, T=30):
    """
    Repair an individual in-place to ensure feasibility.
    For each team, sort its assigned jobs by start day, then push any
    overlapping job to start immediately after the previous job ends.
    Jobs that cannot fit within T after shifting are removed (set to None).
    """
    # Group scheduled jobs by team: {team_id: [engine_id, ...]}
    team_jobs = {team: [] for team in TEAMS}

    for engine_id in range(1, M + 1):
        slot = individual[engine_id - 1]
        if slot is not None:
            team_id, start_date = slot
            team_jobs[team_id].append(engine_id)

    for team_id, job_engine_ids in team_jobs.items():
        # skip teams with no jobs
        if not job_engine_ids:
            continue
        # Sort jobs for this team by their current start day
        job_engine_ids.sort(key=lambda eid: individual[eid - 1][1])

        earliest_available = 1  # next day this team is free to start new job

        for engine_id in job_engine_ids:
            team_id_slot, start_day = individual[engine_id - 1]
            duration = get_maintenance_duration(engine_id, get_team_type(team_id_slot))

            # Push start day forward if team is still busy
            repaired_start = max(start_day, earliest_available)
            repaired_end = repaired_start + duration - 1

            if repaired_end > T:
                # Job cannot fit in horizon; remove it
                individual[engine_id - 1] = None
            else:
                individual[engine_id - 1] = (team_id_slot, repaired_start)
                earliest_available = repaired_end + 1

    return individual

def crossover(parent1, parent2, T=30):
    """
    Single-point crossover on fixed-length individuals of length M.
    A random crossover point is chosen in [1, M-1].
    """
    crossover_point = random.randint(1, M - 1)

    child1 = parent1[:crossover_point] + parent2[crossover_point:]
    child2 = parent2[:crossover_point] + parent1[crossover_point:]

    # Repair both children to restore feasibility
    child1 = repair_individual(child1, T)
    child2 = repair_individual(child2, T)

    return child1, child2

def mutate(individual, engines_in_horizon, T=30):
    """
    Apply one of three mutation operators to a randomly selected engine slot:
    1. Shift start day: move start day by a small random offset (+/- 3 days).
    2. Swap team: reassign the engine to a different randomly chosen team.
    3. Toggle schedule: add an unscheduled engine or remove a scheduled one.
    The individual is repaired after mutation to restore feasibility.
    """
    # Pick a random engine from the horizon
    engine_id = random.choice(engines_in_horizon)
    slot = individual[engine_id - 1]

    operator = random.choice(["shift", "swap_team", "toggle"])

    if operator == "shift" and slot is not None:
        # Move start day by a small random amount
        team_id, start_day = slot
        duration = get_maintenance_duration(engine_id, get_team_type(team_id))
        latest_start = T - duration + 1
        shift = random.randint(-3, 3)
        new_start = max(1, min(latest_start, start_day + shift))
        individual[engine_id - 1] = (team_id, new_start)

    elif operator == "swap_team" and slot is not None:
        # Reassign to a different team
        team_id, start_day = slot
        new_team = random.choice([t for t in TEAMS if t != team_id])
        duration = get_maintenance_duration(engine_id, get_team_type(new_team))
        latest_start = T - duration + 1
        if latest_start >= 1:
            new_start = min(start_day, latest_start)
            individual[engine_id - 1] = (new_team, new_start)

    else:
        # Toggle: add if unscheduled, remove if already scheduled
        if slot is None:
            team_id = random.choice(TEAMS)
            duration = get_maintenance_duration(engine_id, get_team_type(team_id))
            latest_start = T - duration + 1
            if latest_start >= 1:
                start_day = random.randint(1, latest_start)
                individual[engine_id - 1] = (team_id, start_day)
        else:
            individual[engine_id - 1] = None

    # Repair 
    individual = repair_individual(individual, T)
    return individual


# testing functions

In [68]:
rul = load_and_prepare_rul_predictions("RUL_consultancy_predictions_A3.csv")
safety = compute_safety_due_dates(rul)
engines = get_engines_in_horizon(safety)

d1 = get_maintenance_duration(10,"A")
d2 = get_maintenance_duration(10,"B")

tt1 = get_team_type("T1")
tt2 = get_team_type("T2")

pc1 = compute_penalty_cost(1,15,safety[1])
pc2 = compute_penalty_cost(2,None,safety[2])

ind = create_individual(engines)
pop = initialize_population(engines,5)

f = is_feasible(ind)

fit = compute_total_fitness(ind,safety,engines)

parents = parent_selection(pop,[compute_total_fitness(i,safety,engines) for i in pop],2)

repaired = repair_individual(ind)

c1,c2 = crossover(pop[0],pop[1])

mut = mutate(ind,engines)

print("rul",rul)
print("safety",safety)
print("engines",engines)
print("durations",d1,d2)
print("team types",tt1,tt2)
print("penalties",pc1,pc2)
print("individual",ind)
print("population size",len(pop))
print("feasible",f)
print("fitness",fit)
print("parents",parents)
print("repaired",repaired)
print("children",c1,c2)
print("mutated",mut)



rul {1: 135, 2: 125, 3: 63, 4: 100, 5: 103, 6: 122, 7: 106, 8: 90, 9: 121, 10: 67, 11: 101, 12: 89, 13: 87, 14: 122, 15: 114, 16: 101, 17: 52, 18: 33, 19: 84, 20: 10, 21: 63, 22: 141, 23: 119, 24: 26, 25: 173, 26: 128, 27: 70, 28: 96, 29: 96, 30: 87, 31: 14, 32: 54, 33: 128, 34: 8, 35: 8, 36: 24, 37: 21, 38: 58, 39: 144, 40: 29, 41: 23, 42: 13, 43: 67, 44: 146, 45: 100, 46: 53, 47: 130, 48: 151, 49: 14, 50: 100, 51: 106, 52: 34, 53: 34, 54: 126, 55: 174, 56: 18, 57: 102, 58: 38, 59: 113, 60: 112, 61: 23, 62: 54, 63: 75, 64: 24, 65: 152, 66: 18, 67: 174, 68: 13, 69: 130, 70: 90, 71: 130, 72: 59, 73: 113, 74: 115, 75: 115, 76: 3, 77: 27, 78: 165, 79: 82, 80: 84, 81: 6, 82: 11, 83: 182, 84: 53, 85: 142, 86: 113, 87: 126, 88: 117, 89: 111, 90: 28, 91: 29, 92: 24, 93: 51, 94: 55, 95: 143, 96: 140, 97: 109, 98: 87, 99: 127, 100: 24}
safety {1: 135, 2: 125, 3: 63, 4: 100, 5: 103, 6: 122, 7: 106, 8: 90, 9: 121, 10: 67, 11: 101, 12: 89, 13: 87, 14: 122, 15: 114, 16: 101, 17: 52, 18: 33, 19: 84,

In [79]:
rul = {i: 10 for i in range(1, M+1)}
safety = compute_safety_due_dates(rul)
engines = get_engines_in_horizon(safety)

population_size = 5
generations = 5

population = initialize_population(engines, population_size)
fitnesses = [compute_total_fitness(ind, safety, engines) for ind in population]

for g in range(generations):
    parents = parent_selection(population, fitnesses, population_size)
    new_pop = []
    for i in range(0, population_size, 2):
        p1 = parents[i]
        p2 = parents[(i+1) % population_size]
        c1, c2 = crossover(p1, p2)
        c1 = mutate(c1, engines)
        c2 = mutate(c2, engines)
        new_pop.append(c1)
        new_pop.append(c2)
    population = new_pop
    fitnesses = [compute_total_fitness(ind, safety, engines) for ind in population]
    print(g, min(fitnesses))

best = population[fitnesses.index(min(fitnesses))]
print(best)
print(min(fitnesses))
print(is_feasible(best))


0 314316
1 313816
2 314316
3 314316
4 313312
[None, None, ('T1', 20), None, None, None, None, None, ('T4', 21), None, ('T4', 10), ('T3', 13), ('T2', 1), None, None, None, None, None, None, None, None, None, None, ('T3', 10), ('T4', 8), None, None, None, ('T1', 25), None, None, None, None, ('T1', 5), None, None, None, None, None, None, ('T2', 19), ('T1', 28), None, None, None, None, None, None, ('T2', 25), None, None, None, None, ('T3', 22), None, ('T1', 1), None, None, ('T2', 12), None, None, None, None, None, None, None, ('T1', 12), None, None, ('T1', 8), None, ('T3', 1), None, None, ('T3', 18), None, None, None, ('T1', 16), None, None, None, None, ('T2', 5), ('T3', 25), None, None, None, None, None, ('T4', 1), ('T3', 5), None, None, None, None, None, ('T4', 14), None, None]
313312
True
